In [ ]:
import pandas as pd
from datetime import datetime
import numpy as np
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
import os

# Google Sheets Constants
SPREADSHEET_ID = '1BDogl-SCyOvYtoeG2heT9z2UdFlroaGpHYpGgleM9y8'
SHEET_NAMES = {
    'manual': 'no_transcript',  # Manual transcription
    'base': 'base_transcript',    # Base model
    'finetuned': 'ft_transcript'  # Fine-tuned model
}

def get_google_sheets_service():
    SCOPES = ['https://www.googleapis.com/auth/spreadsheets.readonly']
    creds = None
    if os.path.exists('token.json'):
        creds = Credentials.from_authorized_user_file('token.json', SCOPES)
    if not creds or not creds.valid:
        flow = InstalledAppFlow.from_client_secrets_file('credentials.json', SCOPES)
        creds = flow.run_local_server(port=0)
        with open('token.json', 'w') as token:
            token.write(creds.to_json())
    return build('sheets', 'v4', credentials=creds)

def parse_time(time_str):
    return datetime.strptime(time_str, '%H:%M:%S').time()

def calculate_duration_seconds(start_time, end_time):
    start = parse_time(start_time)
    end = parse_time(end_time)
    duration = datetime.combine(datetime.today(), end) - datetime.combine(datetime.today(), start)
    return duration.total_seconds()

def read_sheet_data(service, sheet_name):
    range_name = f'{sheet_name}!A:G'  # Include up to column G to get all data
    print(f"Reading from sheet: {range_name}")
    result = service.spreadsheets().values().get(
        spreadsheetId=SPREADSHEET_ID,
        range=range_name
    ).execute()
    
    rows = result.get('values', [])
    if not rows:
        return pd.DataFrame()
    
    df = pd.DataFrame(rows[1:], columns=rows[0])
    # Clean any empty rows
    df = df.dropna(subset=['transcribing_duration', 'human_transcript'])
    
    # Convert duration string to seconds, handling the rollover at 60 minutes
    def convert_duration(duration_str):
        try:
            m, s = map(int, duration_str.split(':')[:2])  # Only take minutes and seconds
            total_minutes = m
            # If minutes is less than the previous value, it means we've rolled over
            if m < 60:
                total_minutes = m
            else:
                # Convert from MMSS format to actual minutes
                total_minutes = (m // 100) * 60 + (m % 100)
            return total_minutes * 60 + s
        except:
            return 0
    
    df['transcribing_duration_seconds'] = df['transcribing_duration'].apply(convert_duration)
    df['transcript_length'] = df['human_transcript'].str.len()
    df['chars_per_second'] = df['transcript_length'] / df['transcribing_duration_seconds']
    

    return df

def analyze_results():
    service = get_google_sheets_service()
    results = {}
    
    # Read data from all sheets
    for method, sheet_name in SHEET_NAMES.items():
        results[method] = read_sheet_data(service, sheet_name)
    
    # Calculate statistics
    stats = {
        'method': [],
        'total_time_minutes': [],
        'avg_time_per_segment': [],
        'chars_per_minute': [],
        'total_characters': [],
        'segments_count': []
    }
    
    for method, df in results.items():
        stats['method'].append(method)
        stats['total_time_minutes'].append(df['transcribing_duration_seconds'].sum() / 60)
        stats['avg_time_per_segment'].append(df['transcribing_duration_seconds'].mean())
        stats['chars_per_minute'].append((df['chars_per_second'] * 60).mean())
        stats['total_characters'].append(df['transcript_length'].sum())
        stats['segments_count'].append(len(df))
    
    stats_df = pd.DataFrame(stats)
    
    # Calculate speed improvements
    manual_time = stats_df.loc[stats_df['method'] == 'manual', 'total_time_minutes'].values[0]
    stats_df['time_saved_vs_manual'] = manual_time - stats_df['total_time_minutes']
    stats_df['speed_improvement_percent'] = (
        (manual_time - stats_df['total_time_minutes']) / manual_time * 100
    )
    
    return stats_df

def generate_report():
    stats = analyze_results()
    
    print("\n=== Transcription Speed Test Results ===\n")
    
    # Print overall statistics
    print("Overall Statistics:")
    print("-" * 80)
    for _, row in stats.iterrows():
        method = row['method'].title()
        print(f"\n{method} Transcription:")
        print(f"Total time: {row['total_time_minutes']:.2f} minutes")
        print(f"Average time per segment: {row['avg_time_per_segment']:.2f} seconds")
        print(f"Characters per minute: {row['chars_per_minute']:.2f}")
        print(f"Total characters: {row['total_characters']:.0f}")
        print(f"Number of segments: {row['segments_count']}")
        
        if row['method'] != 'manual':
            print(f"Time saved vs manual: {row['time_saved_vs_manual']:.2f} minutes")
            print(f"Speed improvement: {row['speed_improvement_percent']:.1f}%")
    
    # Save detailed results to CSV
    stats.to_csv('transcription_speed_analysis.csv', index=False)
    print("\nDetailed results saved to 'transcription_speed_analysis.csv'")

if __name__ == '__main__':
    generate_report()


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

def read_transcriber_results(result_dir):
    results = []
    for csv_file in Path(result_dir).glob('*transcription_speed_analysis.csv'):
        df = pd.read_csv(csv_file)
        transcriber_id = csv_file.stem.split('_')[1]
        df['transcriber'] = f'Transcriber {transcriber_id}'
        results.append(df)
    return pd.concat(results, ignore_index=True)

def calculate_aggregate_stats(df):
    # Group by method and calculate mean, std, min, max
    stats = df.groupby('method').agg({
        'total_time_minutes': ['mean', 'std', 'min', 'max'],
        'chars_per_minute': ['mean', 'std', 'min', 'max'],
        'speed_improvement_percent': ['mean', 'std', 'min', 'max']
    }).round(2)
    
    # Flatten column names
    stats.columns = [f'{col[0]}_{col[1]}' for col in stats.columns]
    stats = stats.reset_index()
    return stats

def generate_transcriber_comparison(df):
    # Pivot table to compare transcribers across methods
    comparison = df.pivot_table(
        index='transcriber',
        columns='method',
        values=['chars_per_minute', 'speed_improvement_percent'],
        aggfunc='mean'
    ).round(2)
    
    # Flatten column names
    comparison.columns = [f'{col[0]}_{col[1]}' for col in comparison.columns]
    comparison = comparison.reset_index()
    return comparison

def generate_combined_method_report(df):
    # Combine all transcriber data and group by method
    combined = df.groupby('method').agg({
        'total_time_minutes': 'sum',
        'total_characters': 'sum',
        'segments_count': 'sum'
    }).reset_index()
    
    # Calculate overall metrics
    combined['chars_per_minute'] = combined['total_characters'] / combined['total_time_minutes']
    
    # Calculate speed improvements
    manual_time = combined.loc[combined['method'] == 'manual', 'total_time_minutes'].iloc[0]
    combined['time_saved_vs_manual'] = manual_time - combined['total_time_minutes']
    combined['speed_improvement_percent'] = (combined['time_saved_vs_manual'] / manual_time) * 100
    
    return combined.round(2)

def main():
    # Read all results
    result_dir = Path('/home/gangagyatso/Downloads/stt_transcription_speed_test/test_result')
    df = read_transcriber_results(result_dir)
    
    # Calculate aggregate statistics
    stats = calculate_aggregate_stats(df)
    
    # Generate transcriber comparison
    comparison = generate_transcriber_comparison(df)
    
    # Generate combined method report
    combined = generate_combined_method_report(df)
    
    # Save results
    # Save results with descriptive headers
    stats.to_csv(result_dir / 'garchen_rinpoche_aggregate_stats.csv', index=False)
    comparison.to_csv(result_dir / 'garchen_rinpoche_transcriber_comparison.csv', index=False)
    combined.to_csv(result_dir / 'garchen_rinpoche_combined_method_report.csv', index=False)
    
    # Also save a summary text report
    with open(result_dir / 'garchen_rinpoche_transcription_summary.txt', 'w') as f:
        f.write("Transcription Speed Analysis: Garchen Rinpoche's Teachings\n")
        f.write("Audio Transcription Performance Evaluation Report\n")
        f.write("================================================================\n\n")
        f.write("Dataset: Selected segments from Garchen Rinpoche's teachings\n")
        f.write(f"Number of transcribers: {len(comparison)}\n")
        f.write(f"Total segments per transcriber: {df['segments_count'].iloc[0]}\n")
        f.write(f"Total characters transcribed: {combined['total_characters'].sum()}\n")
    
    # Print summary report
    print("\n================================================================")
    print("Transcription Speed Analysis: Garchen Rinpoche's Teachings")
    print("Audio Transcription Performance Evaluation Report")
    print("================================================================")
    
    print("\n=== Aggregate Statistics ===")
    print("\nAverage performance across all transcribers:")
    for _, row in stats.iterrows():
        method = row['method']
        print(f"\n{method.title()} Method:")
        print(f"Average time: {row['total_time_minutes_mean']:.2f} minutes (±{row['total_time_minutes_std']:.2f})")
        print(f"Characters per minute: {row['chars_per_minute_mean']:.2f} (±{row['chars_per_minute_std']:.2f})")
        if method != 'manual':
            print(f"Speed improvement: {row['speed_improvement_percent_mean']:.1f}% (±{row['speed_improvement_percent_std']:.1f}%)")
    
    print("\n=== Transcriber Comparison ===")
    print("\nCharacters per minute for each transcriber:")
    for _, row in comparison.iterrows():
        print(f"\n{row['transcriber']}:")
        print(f"Manual: {row['chars_per_minute_manual']:.2f}")
        print(f"Base: {row['chars_per_minute_base']:.2f}")
        print(f"Finetuned: {row['chars_per_minute_finetuned']:.2f}")
        print(f"Speed improvement with finetuned: {row['speed_improvement_percent_finetuned']:.1f}%")
    
    print("\n=== Combined Method Report (All Transcribers) ===")
    print("\nTotal performance across all transcribers combined:")
    for _, row in combined.iterrows():
        method = row['method']
        print(f"\n{method.title()} Method:")
        print(f"Total time: {row['total_time_minutes']:.2f} minutes")
        print(f"Total characters: {row['total_characters']}")
        print(f"Characters per minute: {row['chars_per_minute']:.2f}")
        if method != 'manual':
            print(f"Total time saved vs manual: {row['time_saved_vs_manual']:.2f} minutes")
            print(f"Overall speed improvement: {row['speed_improvement_percent']:.1f}%")

if __name__ == '__main__':
    main()



Transcription Speed Analysis: Garchen Rinpoche's Teachings
Audio Transcription Performance Evaluation Report

=== Aggregate Statistics ===

Average performance across all transcribers:

Base Method:
Average time: 25.66 minutes (±5.56)
Characters per minute: 55.83 (±11.30)
Speed improvement: -6.3% (±30.5%)

Finetuned Method:
Average time: 16.39 minutes (±5.87)
Characters per minute: 102.46 (±56.12)
Speed improvement: 34.9% (±15.8%)

Manual Method:
Average time: 26.54 minutes (±13.06)
Characters per minute: 57.18 (±16.76)

=== Transcriber Comparison ===

Characters per minute for each transcriber:

Transcriber 1:
Manual: 55.40
Base: 54.69
Finetuned: 94.57
Speed improvement with finetuned: 34.4%

Transcriber 2:
Manual: 37.70
Base: 42.27
Finetuned: 57.79
Speed improvement with finetuned: 47.1%

Transcriber 3:
Manual: 78.63
Base: 69.87
Finetuned: 183.56
Speed improvement with finetuned: 45.4%

Transcriber 4:
Manual: 56.99
Base: 56.47
Finetuned: 73.92
Speed improvement with finetuned: 12.7%